<a href="https://colab.research.google.com/github/aurorafinetti/progetto.tecnologiedeidatiedellinguaggio/blob/main/codici.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas ipywidgets

import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import defaultdict
import time
import os

# ── ELIMINA VECCHIO FILE CSV ──────────────────────────────────────────
if os.path.exists("discogs_nostalgia_2025.csv"):
    os.remove("discogs_nostalgia_2025.csv")
    print("🗑️ Vecchio file CSV eliminato.")

# ── CONFIGURAZIONE ────────────────────────────────────────────────────
TOKEN = "YmYWIbBkQVihZyWWxXIAPhvMZjSeWeRGSxPGOZnx"

headers = {
    "Authorization": f"Discogs token={TOKEN}",
    "User-Agent": "ProgettoVinili/1.0"
}

# ── RACCOLTA DATI DA DISCOGS ──────────────────────────────────────────
print("🔍 Recupero dati da Discogs...")

dati = []
for pagina in range(1, 6):
    print(f"  Scarico pagina {pagina}/5...")
    url = "https://api.discogs.com/database/search"
    params = {
        "type": "release",
        "format": "Vinyl",
        "country": "US",
        "year": 2025,
        "sort": "have",
        "sort_order": "desc",
        "per_page": 50,
        "page": pagina
    }
    response = requests.get(url, headers=headers, params=params)
    risultati = response.json().get("results", [])

    for r in risultati:
        titolo_completo = r.get("title", "N/A")
        # Discogs restituisce il titolo nel formato "Artista - Album"
        if " - " in titolo_completo:
            artista = titolo_completo.split(" - ")[0].strip()
            album = titolo_completo.split(" - ", 1)[1].strip()
        else:
            artista = "N/A"
            album = titolo_completo

        anno = r.get("year", "N/A")
        try:
            anno_int = int(anno)
            categoria = "Contemporaneo" if anno_int >= 2020 else "Nostalgia"
        except:
            categoria = "N/A"

        dati.append({
            "artista": artista,
            "album": album,
            "anno": anno,
            "categoria": categoria
        })
    time.sleep(1)

df = pd.DataFrame(dati)
df = df.reset_index(drop=True)
print(f"✅ Trovati {len(df)} vinili.\n")

# Salva in CSV
df.to_csv("discogs_nostalgia_2025.csv", index=False)

# ── TABELLA COMPLETA ──────────────────────────────────────────────────
def genera_tabella(dataframe, titolo=""):
    righe_html = ""
    colore_alt = False
    for _, r in dataframe.iterrows():
        bg = "#f9f9f9" if colore_alt else "#ffffff"
        colore_alt = not colore_alt
        colore_cat = "#1db954" if r["categoria"] == "Contemporaneo" else "#e85d4a" if r["categoria"] == "Nostalgia" else "#888"
        righe_html += (
            f"<tr style='background:{bg}'>"
            f"<td style='padding:5px 12px'>{r['artista']}</td>"
            f"<td style='padding:5px 12px'>{r['album']}</td>"
            f"<td style='padding:5px 12px; text-align:center'>{r['anno']}</td>"
            f"<td style='padding:5px 12px; text-align:center; color:{colore_cat}; font-weight:bold'>{r['categoria']}</td>"
            f"</tr>"
        )
    return HTML(f"""
    <h3>🎵 {titolo}</h3>
    <p style='color:gray'>Totale: <b>{len(dataframe)}</b> vinili</p>
    <div style='max-height:500px; overflow-y:auto'>
    <table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px; width:100%'>
      <thead style='background:#1a1a2e; color:white; position:sticky; top:0'>
        <tr>
          <th style='padding:7px 12px'>Artista</th>
          <th style='padding:7px 12px'>Album</th>
          <th style='padding:7px 12px'>Anno</th>
          <th style='padding:7px 12px'>Categoria</th>
        </tr>
      </thead>
      <tbody>{righe_html}</tbody>
    </table>
    </div>
    """)

display(genera_tabella(df, "Vinili Discogs – Nostalgia vs Contemporaneo (USA 2025)"))

# ── CONTEGGIO NOSTALGIA VS CONTEMPORANEO ─────────────────────────────
conteggio_cat = df["categoria"].value_counts()

righe_cat = "".join(
    f"<tr><td style='padding:4px 16px'><b>{cat}</b></td>"
    f"<td style='padding:4px 16px; text-align:center'>{'🎵' * min(n, 10)} {n}</td></tr>"
    for cat, n in conteggio_cat.items()
)

display(HTML(f"""
<h3>📊 Nostalgia vs Contemporaneo</h3>
<table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px'>
  <thead style='background:#1a1a2e; color:white'>
    <tr>
      <th style='padding:7px 16px'>Categoria</th>
      <th style='padding:7px 16px'>N° vinili</th>
    </tr>
  </thead>
  <tbody>{righe_cat}</tbody>
</table>
"""))

# ── CONTEGGIO PER ANNO ────────────────────────────────────────────────
conteggio_anno = df[df["anno"] != "N/A"]["anno"].value_counts().sort_index()

righe_anno = "".join(
    f"<tr><td style='padding:4px 16px; text-align:center'><b>{anno}</b></td>"
    f"<td style='padding:4px 16px; text-align:center'>{'🎵' * min(n, 10)} {n}</td></tr>"
    for anno, n in conteggio_anno.items()
)

display(HTML(f"""
<h3>📊 Distribuzione per anno di pubblicazione</h3>
<table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px'>
  <thead style='background:#1a1a2e; color:white'>
    <tr>
      <th style='padding:7px 16px'>Anno</th>
      <th style='padding:7px 16px'>N° vinili</th>
    </tr>
  </thead>
  <tbody>{righe_anno}</tbody>
</table>
"""))

# ── RICERCA INTERATTIVA ───────────────────────────────────────────────
testo_input = widgets.Text(
    value="Nostalgia",
    description="🔎 Cerca:",
    placeholder="Artista, album, anno o categoria",
    layout=widgets.Layout(width="420px")
)
btn = widgets.Button(description="Cerca", button_style="primary", icon="search")
output = widgets.Output()

def cerca(b):
    output.clear_output()
    query = testo_input.value.strip()
    with output:
        if not query:
            print("⚠️ Inserisci un artista, album, anno o categoria.")
            return

        q = query.lower()
        risultati = df[
            df["artista"].str.lower().str.contains(q, na=False) |
            df["album"].str.lower().str.contains(q, na=False) |
            df["anno"].str.lower().str.contains(q, na=False) |
            df["categoria"].str.lower().str.contains(q, na=False)
        ]

        if risultati.empty:
            display(HTML(f"<span style='color:red'>❌ Nessun risultato per '<b>{query}</b>'.</span>"))
            return

        display(genera_tabella(risultati, f"Risultati per \"{query}\""))

btn.on_click(cerca)
display(HTML("<h3>🔎 Filtra per artista, album, anno o categoria</h3>"))
display(widgets.HBox([testo_input, btn]), output)